# Multi-Session ROI Alignment

**What this notebook does.** Defines ROIs on a template video frame, then registers the
ROI grid onto every other session's video (by aligning the first frames of each video
together) so point tracking in each session uses anatomically matched points.

**Pipeline.** Example frames -> `fr.alignment.Image_preparation_pipeline` (VQT spectral
variance + CLAHE) -> `fr.alignment_multisession.Aligner` (RoMa / ECC_cv2 /
PhaseCorrelation / NullRegistration backends) or the built-in `fr.rois.Image_Aligner`
(ECC-only fallback) -> remap ROI points -> save per-session `ROIs.h5` files.

**Inputs.** A directory of videos organized one-folder-per-session (path structure is set
by you in the USER CONFIG cell).

**Outputs.** Per-session `ROIs.h5` files plus a saved template image, and an AVI preview
of the aligned images with warped points overlaid.

## Optional dependencies

- **face-rhythm[multisession]** (`pip install face-rhythm[multisession]`): installs
  `romatch-roicat`, which unlocks the high-accuracy RoMa backend via
  `fr.alignment_multisession.Aligner(method='RoMa', ...)`. Without it, the notebook
  still runs with the always-available `ECC_cv2` / `PhaseCorrelation` /
  `NullRegistration` backends, or the built-in `fr.rois.Image_Aligner` (OpenCV ECC:
  translation/affine/euclidean/homography) --- adequate for near-rigid drift.
- **natsort**: natural-order sorting of session names.
- **torch + CUDA**: RoMa is meaningfully faster on a GPU.


# Set ROIs

This notebook allows you to make ROIs for a video without going through the entire pipeline. The example frame used as the image to set the ROIs is fetched using OpenCV's video reader.\
The functionality of this notebook is technically subsumed by the basic pipeline notebook.

**If your data is local**: Just specify the path to the video you want to draw ROIs on.

**If your data is on a server**: OpenCV's video reader allows for you to read just a single frame from a video file without transferring the entire thing IF the server directory is mounted. So we recommend mounting the server directory if possible to avoid transferring the entire file. This can be done with:
- Windows: https://support.microsoft.com/en-us/windows/map-a-network-drive-in-windows-29ce55d1-34e3-a7e2-4801-131475f9557d
- OSX: https://www.google.com/search?q=mount+network+drive+osx
- Linux: Use RClone CLI: `rclone mount remote:path/to/files /path/to/local/mount`
    - Example: `rclone mount transfer:/n/files/Neurobio/MICROSCOPE/ /mnt/MICROSCOPE/`
    

In [ ]:
# ALWAYS RUN THIS CELL
# widen jupyter notebook window
from IPython.display import display, HTML
display(HTML("<style>.container {width:95% !important; }</style>"))

%load_ext autoreload
%autoreload 2
import face_rhythm as fr
import face_rhythm.alignment

from pprint import pprint
from pathlib import Path
import copy
import getpass

import cv2
import skimage

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import natsort

fr.util.system_info(verbose=True);


## Set paths

In [ ]:
# ======== USER CONFIG ========
from pathlib import Path

## Base directory containing per-mouse subfolders of session-specific videos.
## Layout expected: <base>/<mouse_id>/<session_id>/*.avi
DIR_DEMO_BASE = Path(
    "/path/to/your/data"
)

## Directory containing per-session video subfolders.
DIR_VIDEOS = DIR_DEMO_BASE

## Directory where per-session ROIs.h5 files and the template image will be saved.
DIR_SAVE = DIR_DEMO_BASE / "demo_roi_alignment_out"

## Optional path to a pre-existing template ROIs.h5 to load instead of drawing
## new ROIs interactively. Set to None to draw new ROIs in the GUI.
## A pre-aligned template for session 2673/20230630_130235 is shipped with the demo data.
PATH_TEMPLATE_ROIS = DIR_DEMO_BASE / "ROIs_aligned" / "2673__20230630_130235" / "ROIs.h5"

## Regex matched against filenames when searching DIR_VIDEOS recursively.
## The demo dataset filenames end in `.avi`.
FILENAME_REGEX = r".*\.avi$"

## Optional substring required to appear somewhere in the matched file path
## (useful to restrict the search to one mouse/cohort). Set to None to disable.
STRMATCH_IN_PATH = None  ## e.g. "2673"

## How many directory levels deep to search from DIR_VIDEOS.
DEPTH_SEARCH = 5

## Convert to strings for face_rhythm APIs that expect str paths.
directory_videos = str(DIR_VIDEOS)
directory_save = str(DIR_SAVE)
filename_strMatch = FILENAME_REGEX
strMatch_in_path = STRMATCH_IN_PATH
## =============================


Scan the video directory and list every session video matching the filename regex.

In [ ]:
paths_videos = fr.helpers.find_paths(
    dir_outer=directory_videos,
    reMatch=filename_strMatch,  ## regex to match files inside DIR_VIDEOS
    reMatch_in_path=strMatch_in_path,  ## optional substring required in the full path
    depth=DEPTH_SEARCH,
    verbose=True,
)
paths_videos


Derive a short session name for each video from its parent folder (natural-sorted).

In [ ]:
names_sessions = natsort.natsorted([Path(p).parts[-2].replace('-', '') for p in paths_videos])
names_sessions

Build the image-preparation pipeline (downsample + VQT spectral-variance + CLAHE) used to enhance frames before alignment.

In [ ]:
pipeline = fr.alignment.Image_preparation_pipeline(
    ds_factor=20,
    ptile_specVar_keep=10,
    ptile_intensity_keep=100,
    params_vqt={
        "Fs_sample": 250,
        "Q_lowF": 3.5,
        "Q_highF": 20,
        "F_min": 0.5,
        "F_max": 60,
        "n_freq_bins": 50,
        "window_type": 'hann',
        "downsample_factor": 10,
        "fft_conv": True,
        "plot_pref": False,
    },
    clip_limit=2.0,
    grid_size=60,
    verbose=True,
)

# Define ROIs

Either select new ROIs (`select_mode='gui'`), or import existing ROIs (`path_file=path_to_ROIs.h5_file`).\
Typically, you should make 1 or 2 ROIs. One for defining where the face points should be and one for cropping the frame.

In [ ]:
## Load the first frame of the first discovered video to use as the template image.
## `exampleImage` is required for ``select_mode='gui'`` (asserted in rois.py).
## In ``select_mode='file'`` the template image stored in the h5 overrides this,
## so passing it unconditionally is safe in both branches.
_cap_template = cv2.VideoCapture(str(paths_videos[0]))
_ok_template, _frame_template = _cap_template.read()
_cap_template.release()
assert _ok_template, f"FR ERROR: could not read first frame of {paths_videos[0]}"
## cv2 returns BGR; convert to a float32 single-channel image for the ROI GUI.
exampleImage = cv2.cvtColor(_frame_template, cv2.COLOR_BGR2GRAY).astype(np.float32)

rois = fr.rois.ROIs(
    ## Draw new ROIs in a GUI (default) or load from file if PATH_TEMPLATE_ROIS is set.
    select_mode='gui' if PATH_TEMPLATE_ROIS is None else 'file',
    exampleImage=exampleImage,
    path_file=str(PATH_TEMPLATE_ROIS) if PATH_TEMPLATE_ROIS is not None else None,
    verbose=2,
)


Preview the ROIs on the template image.

In [ ]:
rois.plot_rois()


Save the template ROIs to `<DIR_SAVE>/ROIs.h5`.

In [ ]:
path_save = str(Path(directory_save) / 'ROIs.h5')
rois.save_run_data(path_run_data=path_save, overwrite=True, verbose=1)

# Optional: Multisession alignment

The below code shows how to align the points from one 'template' session onto multiple other 'new' sessions.\
Steps:
1. Get example images from each session: `images`
2. Optionally adjust the local contrast of each example image to make: `images_toUse`
3. Instantiate the `ROI_Aligner` class and choose which OpenCV optical flow method to use: `aligner`
4. Perform non-rigid registration to warp the 'template' ROIs and points onto the example images from each session: `aligner.align_and_make_ROIs`
5. Retrieve the newly made `ROIs` class objects: `rois_objs_new = aligner.ROIs_objects_new`
6. Save the new `ROIs` class objects: `rois_objs_new[x].save_run_data()`
7. Visualize the results!

1. Get example images

Grab a representative frame from each session and run it through the preparation pipeline.

In [ ]:
%matplotlib inline
images = {path: pipeline.apply_pipeline(
    fr.alignment.get_frames(path, time_start=0, time_end=40).mean(-1)
) for path in tqdm(paths_videos)}


3. to 5. Register ROIs from template to new session images, then make new `ROIs` objects

In [ ]:
## Save images
(Path(directory_save) / 'images').mkdir(parents=True, exist_ok=True)
[np.save(str(Path(directory_save) / 'images' / f"{date}.npy"), im) for date, im in tqdm(zip(names_sessions, images.values()))]
;

Convert prepared images to float32 and rescale to [0, 1] so the aligner sees a consistent intensity range.

In [ ]:
ims_aug = [im.astype(np.float32) for im in images.values()]
ia_max = np.max(np.stack(ims_aug, axis=0))
ims_aug = [im / ia_max for im in ims_aug]

Build a contrast-enhanced version of the template image to feed into the aligner.

In [ ]:
im_template = rois.exampleImage

im_aug_template = pipeline.apply_clahe(((im_tmp:=(im_template**0.5)) / im_tmp.max()), clip_limit=2.0, grid_size=25,)

im_aug_template = (im_aug_template.astype(np.float32) / im_aug_template.astype(np.float32).max())

Quick sanity-check — show the enhanced template image.

In [ ]:
plt.figure()
plt.imshow(im_aug_template, cmap='gray')

Scroll through the template and per-session prepared images before alignment.

In [ ]:
fr.visualization.display_toggle_image_stack([im_aug_template] + ims_aug)

Import the multi-session aligner.

In [ ]:
## Use the in-tree multi-session aligner (ported from ROICaT; see
## face_rhythm/alignment_multisession.py). The ``RoMa`` backend requires
## the ``multisession`` extra (``pip install face-rhythm[multisession]``);
## the ``ECC_cv2`` / ``PhaseCorrelation`` / ``NullRegistration`` backends
## are always available.
import face_rhythm.alignment_multisession


Preview the images (optionally blurred and cropped) the aligner will consume.

In [ ]:
fr.visualization.display_toggle_image_stack([skimage.filters.gaussian(im, sigma=.0)[40:-320, 100:-100] for im in ([im_aug_template,] + ims_aug[:])])

Construct the multi-session `Aligner` (uses CUDA if available).

In [ ]:
aligner = face_rhythm.alignment_multisession.Aligner(
    use_match_search=True,
    all_to_all=True,
    radius_in=15,
    radius_out=75,
    order=5,
    z_threshold=50,
    device='cuda:0' if torch.cuda.is_available() else 'cpu',
    verbose=2,
)


Fit the geometric registration from each session image onto the template.

In [ ]:
aligner.fit_geometric(
    ## ``template`` specifies which image to use as the template.
    ## Either array (image), integer (ims_moving index), or float (ims_moving fractional index).
    template=0,
    ims_moving=[skimage.filters.gaussian(im, sigma=4.0) for im in ([im_aug_template[:,:,0],] + ims_aug[:])],
    template_method='image',  ## 'sequential': align images to neighboring images (good for drifting data). 'image': align to a single image
    mask_borders=(0,50, 0,0),  ## number of pixels to mask off the edges (top, bottom, left, right)
    method='RoMa',  ## See below for options. Requires face-rhythm[multisession] for RoMa.
    kwargs_method={
        'RoMa': {  ## Accuracy: Best, Speed: Very slow (can be fast with a GPU). Requires ``multisession`` extra.
            'model_type': 'outdoor',
            'n_points': 10000,  ## Higher values mean more points are used for the registration. Useful for larger FOV_images. Larger means slower.
            'batch_size': 1000,
        },
        'ECC_cv2': {  ## Accuracy: Okay. Speed: Medium.
            'mode_transform': 'euclidean',  ## Must be one of {'translation', 'affine', 'euclidean', 'homography'}. See cv2 documentation on findTransformECC for more details.
            'n_iter': 200,
            'termination_eps': 1e-09,  ## Termination criteria for the registration algorithm. See documentation for more details.
            'gaussFiltSize': 1,  ## Size of the gaussian filter used to smooth the FOV_image before registration. Larger values mean more smoothing.
            'auto_fix_gaussFilt_step': 10,  ## If the registration fails, then the gaussian filter size is reduced by this amount and the registration is tried again.
        },
        'PhaseCorrelation': {  ## Accuracy: Poor. Speed: Very fast. Notes: Only applicable for translations, not rotations or scaling.
            'bandpass_freqs': [1, 30],
            'order': 5,
        },
    },
    constraint='euclidean',
    kwargs_RANSAC={
        'inl_thresh': 3.0,  ## cv2.findHomography RANSAC inlier threshold. Larger values mean more lenient matching.
        'max_iter': 100,
        'confidence': 0.99,
    },
    verbose=True,  ## Set to 3 to view plots of the alignment process if available for the method.
)


Apply the learned remapping to produce aligned images and plot the result.

In [ ]:
ims_aligned = aligner.transform_images(
    ims_moving=[im_aug_template[:,:,0],] + ims_aug,
    remappingIdx=aligner.remappingIdx_geo,
)[1:]

remappingIdx_geo = copy.deepcopy(aligner.remappingIdx_geo[1:])

aligner.plot_alignment_results_geometric()


Scroll through the aligned images — each session's first frame warped onto the template.

In [ ]:
fr.visualization.display_toggle_image_stack([im_aug_template,] + ims_aligned)

Helper — warp (x, y) points using the aligner's remap field and clip to image bounds.

In [ ]:
def transform_points(points, remappingIdx):
    ## Transform points using the remapping index
    points_remap = fr.helpers.remap_points(
        points=points,
        remappingIdx=remappingIdx,
        interpolation='linear',
        fill_value=None,
    )

    ## Clip points to image size
    points_remap[:, 0] = np.clip(points_remap[:, 0], 0, remappingIdx.shape[1] - 1)
    points_remap[:, 1] = np.clip(points_remap[:, 1], 0, remappingIdx.shape[0] - 1)

    return points_remap


Warp each ROI's border points and tracking-point grid from the template onto every session.

In [ ]:
remappingIdx = copy.deepcopy(remappingIdx_geo)

points_roiBorders_transformed = [{key: transform_points(
    points=points,
    remappingIdx=rmap_idx,
) for key, points in rois.roi_points.items()} for rmap_idx in remappingIdx]

points_forTracking_transformed = [transform_points(
    points=rois.point_positions,
    remappingIdx=rmap_idx,
) for rmap_idx in remappingIdx]

exampleImages = [np.tile((e * (0.15)).astype(np.uint8)[..., None], (1, 1, 3)) for e in images.values()]


Build a new per-session `ROIs` object from the warped points.

In [ ]:
rois_objs_new = {name: fr.rois.ROIs(
    select_mode='custom', 
    coords_rois=borderPoints, 
    exampleImage=exampleImage, 
    point_positions=point_positions
) for name, borderPoints, exampleImage, point_positions in tqdm(zip(paths_videos, points_roiBorders_transformed, exampleImages, points_forTracking_transformed))}

Render each session's template frame with the warped tracking points overlaid.

In [ ]:
frame_visualizer = fr.visualization.FrameVisualizer(
    display=False,
    frame_height_width=rois_objs_new[list(rois_objs_new.keys())[0]].img_hw,
    point_sizes=3,
    points_colors=(255,0,0),
    alpha=0.5,
)

images_with_warped_points = [frame_visualizer.visualize_image_with_points(
    image=rois.exampleImage,
    points=rois.point_positions,
) for rois in rois_objs_new.values()]

Visual check — make sure the warped points land on the face across every session.

In [ ]:
fr.visualization.display_toggle_image_stack(images_with_warped_points)

## 6. Save
Save the new `ROIs` objects. These can be used to initialize the `ROIs` objects in each face-rhythm run.

Save each session's `ROIs.h5` under `<DIR_SAVE>/<session_name>/ROIs.h5`.

In [ ]:
for ii, (name, rois_new) in enumerate(rois_objs_new.items()):
    path_save = str(Path(directory_save) / names_sessions[ii] / f'ROIs.h5')
    rois_new.save_run_data(
        path_run_data=path_save,
        overwrite=True,
        verbose=1,
    )


## 7. Visualize!

Save a preview GIF showing each session's image with warped tracking points and a frame number — use this to eyeball alignment quality across sessions.

In [ ]:
## Cast to uint8
movie = [im.astype(np.uint8) for im in images_with_warped_points]
## Add text overlay
movie = fr.helpers.add_text_to_images(movie, [[str(n)] for n in np.arange(len(movie))], position=(50,100), font_size=4, line_width=5,)

image_saver = fr.util.Image_Saver(
    dir_save=directory_save,
    overwrite=True,
)

image_saver.save_gif(
    array_images=movie, 
    name_save='mouse_face_points_matched', 
    frame_rate=6.0, 
    loop=0, 
)